# Data cleaning, step by step

This notebook walks through what `python -m src.get_data` does to the raw
download, one step at a time, using the same functions from `src/get_data.py`.
It does not save anything. The script is still the one that writes
`data/clean/`.

It reads the raw files in `data/raw/`, which are only on the machine that ran
the download. If they are not there it downloads them again, which needs an
internet connection.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Work from the repo root whether the notebook is opened from there or notebooks/
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(root)
sys.path.insert(0, str(root))

from src import config, get_data  # noqa: E402

## 1. Raw data

In [ ]:
if config.RAW_PRICES_FILE.exists():
    raw = pd.read_csv(config.RAW_PRICES_FILE, index_col=0, parse_dates=True)
    raw_rf = pd.read_csv(config.RAW_RISK_FREE_FILE, index_col=0, parse_dates=True)
else:
    raw = get_data.download_prices()
    raw_rf = get_data.download_risk_free()

print(f"Raw prices: {raw.shape[0]} rows, {raw.shape[1]} tickers")
print(f"Raw risk-free: {raw_rf.shape[0]} rows")
raw.head()

In [ ]:
# Missing values per ticker in the raw download
raw.isna().sum()

## 2. Tidy dates and columns

Keep the seven tickers in a fixed order, make the dates plain (no timezone),
drop duplicate dates and sort.

In [ ]:
prices = get_data.tidy(raw)
print(f"{len(raw)} rows -> {len(prices)} rows")
print("Columns:", list(prices.columns))

## 3. Trim to the study period

In [ ]:
before = len(prices)
prices = get_data.trim_dates(prices)
print(f"{before} rows -> {len(prices)} rows")
print(f"From {prices.index[0].date()} to {prices.index[-1].date()}")

## 4. Bad ticks and holidays

A zero or negative price can't be real, so it becomes missing. Rows where no
ticker has a price are exchange holidays and are dropped.

In [ ]:
print("Zero or negative prices:", int((prices <= 0).sum().sum()))
before = len(prices)
prices = get_data.drop_holidays(get_data.remove_bad_ticks(prices))
print(f"{before} rows -> {len(prices)} rows")

## 5. Start on the first day every ticker trades

In [ ]:
before = len(prices)
prices = get_data.start_when_all_trade(prices)
print(f"{before} rows -> {len(prices)} rows, first day {prices.index[0].date()}")

## 6. Fill short gaps

A gap of up to 3 days takes the last price. A longer gap stops the script,
because it means something is wrong with the download.

In [ ]:
print("Missing before filling:")
print(prices.isna().sum().to_string())
prices, filled = get_data.fill_short_gaps(prices)
print("\nDays filled:")
print(filled.to_string())

## 7. Same result as the script

Running the whole cleaning in one call should give exactly the same table.

In [ ]:
script_prices, _ = get_data.clean_prices(raw)
pd.testing.assert_frame_equal(prices, script_prices)
print("Step by step result matches clean_prices()")

## 8. Risk-free rate

FRED does not publish on bond market holidays, some of which are normal
trading days for stocks. Those days take the last published rate.

In [ ]:
risk_free = get_data.clean_risk_free(raw_rf, prices.index)
published = pd.to_datetime(raw_rf.dropna().index)
carried = (~prices.index.isin(published)).sum()
print(f"{len(risk_free)} days, {carried} carried forward from the day before")
risk_free.describe()

## 9. Checks that change nothing

Big daily moves are listed so someone can confirm they are real, and the
stress periods have to sit inside the data.

In [ ]:
moves = get_data.large_moves(prices)
moves.rename("log_return").to_frame()

In [ ]:
for name, (start, end) in config.STRESS_PERIODS.items():
    status = "covered" if get_data.covers_period(prices, start, end) else "NOT COVERED"
    print(f"{name}: {status}")

## 10. Daily log returns

These are what `load_returns()` gives the rest of the project.

In [ ]:
returns = np.log(prices).diff().dropna()
returns.describe().T

In [ ]:
ax = (prices / prices.iloc[0] * 100).plot(logy=True, figsize=(11, 5))
ax.set_title("Prices rebased to 100 (log scale)")
ax.set_ylabel("")